# SchemeXpress — Phase 2: Data Understanding, Quality Analysis, Preprocessing & EDA

**Dataset:** Indian Government Schemes (Kaggle, ~3,400 records)  
**Raw file:** `data/raw/updated_data.csv`  
**Clean output:** `data/processed/cleaned_schemes.csv`  
**Charts:** `docs/_eda_charts/*.png`

---

### What this notebook does

| Section | Purpose |
|---|---|
| 1 | Load & understand the raw dataset |
| 2 | Data quality analysis |
| 3 | Preprocessing |
| 4 | Exploratory Data Analysis (EDA) with charts |
| 5 | Key findings summary |

> **Rule:** `data/raw/updated_data.csv` is **never modified**. All changes go to `data/processed/`.

In [ ]:
import os, sys, re, html, warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 110, 'figure.figsize': (10, 5)})
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_columns', 20)

# Paths — works from any working directory
ROOT     = os.path.abspath(os.path.join(os.getcwd(), '..'))
RAW      = os.path.join(ROOT, 'data', 'raw',       'updated_data.csv')
CLEAN    = os.path.join(ROOT, 'data', 'processed', 'cleaned_schemes.csv')
CHARTS   = os.path.join(ROOT, 'docs', '_eda_charts')

print('Raw  :', RAW)
print('Clean:', CLEAN)

---
## Section 1 — Load & Understand the Raw Dataset

In [ ]:
raw = pd.read_csv(RAW, low_memory=False)
print(f'Shape: {raw.shape[0]:,} rows × {raw.shape[1]} columns')
print(f'\nColumns ({raw.shape[1]}):')
for i, col in enumerate(raw.columns):
    print(f'  {i:>2}. {col!r}')

In [ ]:
# Data types
print('Data types:')
print(raw.dtypes.to_string())

In [ ]:
# First 3 records — transposed for readability
raw.head(3).T

In [ ]:
# Missing values
null_counts = raw.isnull().sum()
null_pct    = (null_counts / len(raw) * 100).round(2)
missing_df  = pd.DataFrame({'missing_count': null_counts, 'missing_%': null_pct})
print('Missing values per column:')
print(missing_df.to_string())

In [ ]:
# Duplicates
exact_dupes = int(raw.duplicated().sum())
slug_dupes  = int(raw.duplicated(subset=['slug']).sum()) if 'slug' in raw.columns else 0
print(f'Exact full-row duplicates : {exact_dupes}')
print(f'Duplicate slug values     : {slug_dupes}')

In [ ]:
# Unique value counts
print('Unique values per column:')
for col in raw.columns:
    print(f'  {col!r:<30}  {raw[col].nunique():>5} unique')

In [ ]:
# Text field length statistics
text_cols = [c for c in ['details','benefits','eligibility','application','documents','tags']
             if c in raw.columns]

print(f'Text field length statistics (non-null, non-blank rows):\n')
print(f'  {"Column":<15}  {"n":>6}  {"min":>6}  {"median":>7}  {"mean":>7}  {"max":>8}')
print('  ' + '-'*55)
for col in text_cols:
    s = raw[col].dropna().astype(str)
    s = s[s.str.strip() != ''].str.len()
    if len(s):
        print(f'  {col:<15}  {len(s):>6,}  {int(s.min()):>6}  '
              f'{int(s.median()):>7,}  {int(s.mean()):>7,}  {int(s.max()):>8,}')

---
## Section 2 — Data Quality Analysis

We check for every type of issue that could affect model quality.

In [ ]:
# Empty / whitespace-only strings (different from NaN)
print('Empty / whitespace-only values per text column:')
for col in text_cols:
    blank = raw[col].fillna('').astype(str).str.strip().eq('').sum()
    nan   = int(raw[col].isnull().sum())
    print(f'  {col:<15}  {blank:>5} blank  (includes {nan} NaN)')

In [ ]:
# BOM and HTML entity contamination
print('BOM chars & HTML entities in text columns:')
for col in text_cols:
    bom = raw[col].dropna().astype(str).str.contains('\ufeff', regex=False).sum()
    ent = raw[col].dropna().astype(str).str.contains(r'&[a-zA-Z]+;|&#\d+;', regex=True).sum()
    print(f'  {col:<15}  BOM in {bom:>5} rows  |  HTML entities in {ent:>4} rows')

In [ ]:
# Scheme names with leading quote characters (CSV scraping artifact)
if 'scheme_name' in raw.columns:
    quoted = raw['scheme_name'].astype(str).str.startswith('"').sum()
    print(f'Scheme names starting with " : {quoted}')
    print('Sample:')
    print(raw[raw['scheme_name'].astype(str).str.startswith('"')]['scheme_name'].head(3).to_list())

In [ ]:
# Categorical value distribution
for col in ['level', 'schemeCategory']:
    if col in raw.columns:
        vc = raw[col].value_counts(dropna=False)
        print(f'\n{col} ({vc.shape[0]} unique values):')
        print(vc.head(20).to_string())

### Data Quality Summary

| Issue | Finding | Action |
|---|---|---|
| Unnamed column (`Unnamed: 9`) | 100% empty | Dropped |
| Exact duplicate rows | 3 rows | Removed (keep first) |
| Duplicate slugs | 0 | No action needed |
| Missing `application` | 2 rows (0.06%) | Filled with `''` |
| Missing `documents` | 11 rows (0.32%) | Filled with `''` |
| Missing `tags` | 29 rows (0.85%) | Filled with `''` |
| BOM characters (`\ufeff`) | 5,045 cell occurrences across all text fields | Stripped |
| Leading quotes on scheme names | 61 rows | Stripped |
| HTML entities (`&amp;` etc.) | 1 row in `details` | Decoded via `html.unescape()` |

---
## Section 3 — Preprocessing

All cleaning is applied to a copy. Raw data is never touched.

In [ ]:
df = raw.copy()

# Step 1: Drop unnamed/empty-header columns
unnamed = [c for c in df.columns
           if str(c).strip() == '' or str(c).startswith('Unnamed:')]
df = df.drop(columns=unnamed)
print(f'Dropped columns        : {unnamed}')
print(f'Shape after drop       : {df.shape}')

# Step 2: Remove exact duplicates
before = len(df)
df = df.drop_duplicates(keep='first')
print(f'Exact dupes removed    : {before - len(df)}')

# Step 3: Remove duplicate slugs
if 'slug' in df.columns:
    before = len(df)
    df = df.drop_duplicates(subset=['slug'], keep='first')
    print(f'Slug dupes removed     : {before - len(df)}')

print(f'Rows after dedup       : {len(df):,}')

In [ ]:
# Step 4: Fill missing values
# Text NLP fields → '' (missing text = no NLP signal, not an error)
TEXT_FILL = [c for c in ['details','benefits','eligibility',
                          'application','documents','tags'] if c in df.columns]
for col in TEXT_FILL:
    df[col] = df[col].fillna('')

if 'schemeCategory' in df.columns:
    df['schemeCategory'] = df['schemeCategory'].fillna('Uncategorized')
if 'level' in df.columns:
    df['level'] = df['level'].fillna('Unknown')

# Derive missing slugs from scheme_name
def _slugify(name):
    s = str(name).lower().strip()
    s = re.sub(r'[^a-z0-9\s-]', '', s)
    s = re.sub(r'\s+', '-', s)
    s = re.sub(r'-+', '-', s)
    return s.strip('-')

if 'slug' in df.columns:
    df['slug'] = df['slug'].fillna(df['scheme_name'].apply(_slugify))

print(f'Null values after fill : {df.isnull().sum().sum()}')

In [ ]:
# Step 5: Clean text (NLP-safe — does NOT lowercase or stem)
def clean_text(text):
    """
    NLP-safe cleaning:
    - Remove BOM, zero-width chars, non-breaking spaces
    - Decode HTML entities (&amp; → &, etc.)
    - Collapse newlines/tabs → single space
    - Collapse multiple spaces → single space
    - Strip leading/trailing whitespace
    Does NOT lowercase or stem — those steps belong in the NLP pipeline (Phase 6).
    """
    if not isinstance(text, str) or text == '':
        return ''
    text = text.replace('\ufeff','').replace('\u200b','').replace('\u00a0',' ')
    text = html.unescape(text)
    text = re.sub(r'[\n\r\t]+', ' ', text)
    text = re.sub(r' {2,}', ' ', text)
    return text.strip()

ALL_TEXT = [c for c in ['scheme_name','details','benefits','eligibility',
                         'application','documents','tags'] if c in df.columns]
for col in ALL_TEXT:
    df[col] = df[col].apply(clean_text)

# Strip outer quotes from scheme_name
df['scheme_name'] = df['scheme_name'].str.strip().str.strip('"').str.strip("'").str.strip()

print('Text cleaning applied.')

In [ ]:
# Step 6: Standardize categoricals
VALID_LEVELS = {'Central', 'State', 'District', 'Unknown'}
if 'schemeCategory' in df.columns:
    df['schemeCategory'] = df['schemeCategory'].str.strip().str.title()
if 'level' in df.columns:
    df['level'] = df['level'].str.strip().str.title()
    df['level'] = df['level'].apply(lambda x: x if x in VALID_LEVELS else 'Unknown')

# Step 7: Build combined_text (for TF-IDF in Phase 6)
ct_parts = [c for c in ['scheme_name','schemeCategory','details',
                          'benefits','eligibility','tags'] if c in df.columns]
df['combined_text'] = df[ct_parts].fillna('').agg(' '.join, axis=1)
df['combined_text'] = df['combined_text'].apply(
    lambda t: re.sub(r' {2,}', ' ', t.strip()))

print(f'Final clean shape      : {df.shape}')
print(f'Columns                : {list(df.columns)}')
print(f'Null values remaining  : {df.isnull().sum().sum()}')

In [ ]:
# Save clean dataset
os.makedirs(os.path.dirname(CLEAN), exist_ok=True)
df.to_csv(CLEAN, index=False, encoding='utf-8')

verify = pd.read_csv(CLEAN)
print(f'Saved → {CLEAN}')
print(f'Reload check: {verify.shape[0]:,} rows × {verify.shape[1]} columns  ✓')

---
## Section 4 — Exploratory Data Analysis

In [ ]:
# Work from the clean dataset
df = pd.read_csv(CLEAN)
print(f'Working dataset: {df.shape[0]:,} rows × {df.shape[1]} columns')

In [ ]:
# ── Chart 1: Level distribution ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
vc = df['level'].value_counts()
colors = sns.color_palette('muted', len(vc))
bars = ax.bar(vc.index, vc.values, color=colors, edgecolor='white')
for bar, val in zip(bars, vc.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f'{val:,}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_title('Distribution of Schemes by Level', fontsize=13, fontweight='bold')
ax.set_xlabel('Level')
ax.set_ylabel('Number of Schemes')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()
print('\nCounts:')
print(vc.to_string())
print('\nInterpretation: State-level schemes dominate the dataset, '
      'reflecting India\'s decentralised welfare delivery.')

In [ ]:
# ── Chart 2: Top 20 scheme categories ────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 7))
top_cats = df['schemeCategory'].value_counts().head(20)
sns.barplot(x=top_cats.values, y=top_cats.index, ax=ax, palette='Blues_d')
for i, val in enumerate(top_cats.values):
    ax.text(val + 3, i, f'{val:,}', va='center', fontsize=9)
ax.set_title('Top 20 Scheme Categories by Count', fontsize=13, fontweight='bold')
ax.set_xlabel('Number of Schemes')
ax.set_ylabel('Scheme Category')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()
print('\nInterpretation: Social Welfare & Empowerment is the dominant category, '
      'followed by Education & Learning and Agriculture-related schemes.')

In [ ]:
# ── Chart 3: Missing values in raw dataset ────────────────────────────────
raw_check = pd.read_csv(RAW)
raw_check = raw_check.drop(columns=[c for c in raw_check.columns
                                     if str(c).strip()=='' or str(c).startswith('Unnamed:')],
                            errors='ignore')
null_pct = (raw_check.isnull().sum() / len(raw_check) * 100).round(2)
null_pct = null_pct[null_pct > 0]

if len(null_pct):
    fig, ax = plt.subplots(figsize=(8, 3))
    bars = ax.barh(null_pct.index, null_pct.values,
                   color=sns.color_palette('Reds_d', len(null_pct)))
    for bar, val in zip(bars, null_pct.values):
        ax.text(val + 0.05, bar.get_y() + bar.get_height()/2,
                f'{val:.2f}%', va='center', fontsize=9)
    ax.set_title('Missing Values (%) per Column — Raw Dataset',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('% Missing')
    ax.set_ylabel('Column')
    ax.spines[['top','right']].set_visible(False)
    plt.tight_layout()
    plt.show()
    print('\nInterpretation: Missing data is sparse (all fields < 1%). '
          'No rows need to be dropped.')
else:
    print('No missing values to chart.')

In [ ]:
# ── Chart 4: Text field length distributions ──────────────────────────────
plot_cols = [c for c in ['details','eligibility','benefits'] if c in df.columns]
fig, axes = plt.subplots(1, len(plot_cols), figsize=(6*len(plot_cols), 5))
if len(plot_cols) == 1:
    axes = [axes]
for ax, col in zip(axes, plot_cols):
    s = df[col].astype(str)
    s = s[s.str.strip() != ''].str.len()
    clip = int(s.quantile(0.99))
    ax.hist(s.clip(upper=clip), bins=50,
            color=sns.color_palette('muted')[2], edgecolor='white', alpha=0.85)
    ax.set_title(f"'{col}'\n(99th pct = {clip:,} chars)",
                 fontsize=10, fontweight='bold')
    ax.set_xlabel('Characters')
    ax.set_ylabel('Schemes')
    ax.spines[['top','right']].set_visible(False)
plt.suptitle('Text Field Length Distributions (Clean Dataset)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print('\nInterpretation: Text lengths are right-skewed. '
      'Most schemes have moderate descriptions; a few have very long text.')

In [ ]:
# ── Chart 5: Top 20 tags ──────────────────────────────────────────────────
if 'tags' in df.columns:
    all_tags = (df['tags'].astype(str)
                .str.split(',').explode()
                .str.strip().str.title())
    all_tags = all_tags[all_tags.str.len() > 1]
    top_tags = all_tags.value_counts().head(20)
    
    fig, ax = plt.subplots(figsize=(12, 7))
    sns.barplot(x=top_tags.values, y=top_tags.index, ax=ax, palette='Greens_d')
    for i, val in enumerate(top_tags.values):
        ax.text(val + 1, i, f'{val:,}', va='center', fontsize=9)
    ax.set_title('Top 20 Most Frequent Tags', fontsize=13, fontweight='bold')
    ax.set_xlabel('Frequency')
    ax.set_ylabel('Tag')
    ax.spines[['top','right']].set_visible(False)
    plt.tight_layout()
    plt.show()
    print('\nInterpretation: "Financial Assistance" is by far the most common tag, '
          'confirming the dataset is strongly focused on direct benefit transfer schemes.')

In [ ]:
# ── Chart 6: Top 10 categories × Level (stacked bar) ─────────────────────
top10 = df['schemeCategory'].value_counts().head(10).index.tolist()
sub   = df[df['schemeCategory'].isin(top10)]
pivot = sub.groupby(['schemeCategory','level']).size().unstack(fill_value=0)
for lv in ['Central','State','District','Unknown']:
    if lv not in pivot.columns: pivot[lv] = 0
pivot = pivot[['Central','State','District','Unknown']]
pivot = pivot.loc[pivot.sum(axis=1).sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(13, 6))
pivot.plot(kind='barh', stacked=True, ax=ax,
           color=['#4878CF','#6ACC65','#D65F5F','#B47CC7'], edgecolor='white')
ax.set_title('Top 10 Categories: Scheme Count by Level',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Number of Schemes')
ax.set_ylabel('Scheme Category')
ax.legend(title='Level', bbox_to_anchor=(1.01,1), loc='upper left')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()
print('\nInterpretation: Agriculture and Business schemes are heavily State-level, '
      'while Education and Skill Development have stronger Central government presence.')

In [ ]:
# ── Chart 7: combined_text length ────────────────────────────────────────
ct = df['combined_text'].str.len()
clip = int(ct.quantile(0.99))
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(ct.clip(upper=clip), bins=60,
        color=sns.color_palette('muted')[0], edgecolor='white', alpha=0.85)
ax.axvline(int(ct.median()), color='crimson', linestyle='--', linewidth=1.5,
           label=f'Median = {int(ct.median()):,} chars')
ax.set_title('Combined Text Length Distribution (clipped at 99th pct)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Characters')
ax.set_ylabel('Number of Schemes')
ax.legend()
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()
print(f'\nMin combined_text length  : {int(ct.min()):,} chars')
print(f'Median combined_text length: {int(ct.median()):,} chars')
print(f'Max combined_text length  : {int(ct.max()):,} chars')
print('\nInterpretation: Most schemes have 1,000–4,000 chars of combined text, '
      'providing sufficient signal for TF-IDF vectorization.')

---
## Section 5 — Key Findings Summary

| Finding | Value |
|---|---|
| Raw dataset dimensions | 3,400 rows × 11 columns |
| Clean dataset dimensions | **3,397 rows × 11 columns** |
| Rows removed | 3 (exact duplicates) |
| Columns removed | 1 (`Unnamed: 9` — 100% empty) |
| Columns added | 1 (`combined_text`) |
| Null values in clean output | 0 |
| BOM characters cleaned | ~5,045 occurrences across text fields |
| Scheme names with leading quotes | 61 (cleaned) |
| State-level schemes | ~2,300+ |
| Central-level schemes | ~1,000+ |
| Most common category | Social Welfare & Empowerment |
| Most common tag | Financial Assistance |
| Median `details` length | ~1,200 chars |
| Median `combined_text` length | ~2,000+ chars |

### Preprocessing decisions

1. **Missing text → `''`** not dropped, because a scheme with a missing `documents` field is still a valid scheme. The NLP engine receives no signal from that field — it is not penalized.
2. **Light text cleaning only** — no lowercasing, no stemming, no stopword removal at this stage. Those steps belong inside the NLP vectorization pipeline (Phase 6) where we can tune them.
3. **`combined_text` built now** — concatenates `scheme_name + schemeCategory + details + benefits + eligibility + tags`. Ready for TF-IDF in Phase 6.

### Next phase

**Phase 3 — Feature Engineering:** Extract structured eligibility features (age range, gender, income category, state, occupation) from the `eligibility` free text field using regex and keyword matching.